# Download PR review comment dataset

Pipeline: GH Archive → filter by top snapshot commits → GitHub compare/zipball enrichment → annotated patched content.

Set `GITHUB_TOKEN` in `.env` (or the environment) before enrichment steps. Install the package editable: `pip install -e .` from the repo root.

In [ ]:
import asyncio
import logging

import aiohttp
from dotenv import load_dotenv

from ai_code_reviewer.dataset import (
    checkpoints,
    gh_archive,
    github_api,
    patches,
)
from ai_code_reviewer.dataset import config as dataset_config

load_dotenv()
logging.basicConfig(level=logging.INFO)

In [ ]:
gh_archive_semaphore = asyncio.Semaphore(dataset_config.GH_ARCHIVE_CONCURRENCY)
gh_semaphore = asyncio.Semaphore(dataset_config.GITHUB_API_CONCURRENCY)

In [ ]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GH_ARCHIVE_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    dataset = await gh_archive.fetch_pr_comments_range(
        session,
        dataset_config.RANGE_START,
        dataset_config.RANGE_END,
        gh_archive_semaphore,
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_RAW_PATH)

In [ ]:
dataset = gh_archive.filter_dataset_by_top_snapshot_commits(
    dataset, dataset_config.SNAPSHOT_COMMITS_TO_KEEP
)

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FILTERED_PATH)

In [ ]:
async with aiohttp.ClientSession(
    connector=dataset_config.tcp_connector_for_concurrency(
        dataset_config.GITHUB_API_CONCURRENCY
    ),
    timeout=dataset_config.default_client_timeout(),
) as session:
    await github_api.enrich_dataset_with_base_and_patches(
        dataset, session, gh_semaphore
    )

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_ENRICHED_PATH)

In [ ]:
patches.enrich_dataset_with_patched_content(dataset)

checkpoints.save_dataset_checkpoint(dataset, dataset_config.DATASET_FINAL_PATH)

In [ ]:
# Compute dataset statistics
num_prs = 0
num_snapshot_commits = 0
num_files = 0
num_files_without_comments = 0
num_comments = 0

for repo_name, pr_map in dataset.items():
    for pr_number, pr_entry in pr_map.items():
        num_prs += 1
        for commit_sha, path_map in pr_entry["commits"].items():
            num_snapshot_commits += 1
            for path, file_entry in path_map.items():
                num_files += 1
                comments = file_entry.get("comments", [])
                num_comments += len(comments)
                if len(comments) == 0:
                    num_files_without_comments += 1

print(f"Number of PRs: {num_prs}")
print(f"Number of snapshot commits: {num_snapshot_commits}")
print(f"Number of files: {num_files}")
print(f"Number of files without comments: {num_files_without_comments}")
print(f"Number of comments in all files: {num_comments}")